In [ ]:
import pandas as pd
import numpy as np

import dagshub
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

dagshub.init(repo_owner='dkhak22', repo_name='ml-assignment-2', mlflow=True)


## Load test data

In [ ]:
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test = test_transaction.merge(test_identity, on='TransactionID', how='left')

transaction_ids = test['TransactionID'].copy()

del test_transaction, test_identity

print(f'Test shape: {test.shape}')


## Load best model from registry

In [ ]:
client = MlflowClient()
versions = client.get_latest_versions('XGBoost_FraudDetection')
latest_version = sorted(versions, key=lambda v: int(v.version))[-1].version

model = mlflow.sklearn.load_model(f'models:/XGBoost_FraudDetection/{latest_version}')
print(f'Loaded XGBoost_FraudDetection version {latest_version}')


## Generate predictions and save submission

In [ ]:
preds = model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud':       preds,
})

output_path = '/kaggle/working/submission.csv'
submission.to_csv(output_path, index=False)

print(f'Saved {len(submission):,} predictions to {output_path}')
print(submission.head(10))
print(f'\nPredicted fraud rate: {preds.mean():.4f}')
